<style>
.jp-RenderedHTMLCommon h1 { color:#ff9900; font-size:2.35em; }
.jp-RenderedHTMLCommon h2 { color:#2563a8; }
.jp-RenderedHTMLCommon h3 { color:#374151; }
.jp-RenderedHTMLCommon blockquote { border-left:6px solid #ff9900; background:#fff7e8; padding:.65em 1em; }
.jp-RenderedHTMLCommon table { font-size:.91em; }
.jp-RenderedHTMLCommon code { color:#9a3412; }
</style>

# Amazon Athena: Introduction
## Serverless, interactive SQL analytics over the data lake

**Architecture · query lifecycle · planning · runtime · results · reuse · security · cost**

> Amazon Athena lets you analyze data in place with SQL while AWS operates the query infrastructure.


# Learning outcomes

After this lesson, you can:

- explain what Athena is and when it fits;
- connect S3 data, the Glue Data Catalog, workgroups, and the query engine;
- trace a query from submission to its distributed execution stages;
- distinguish query planning from query runtime;
- configure and secure the query-results location;
- explain result reuse and its boundaries;
- reduce scanned data, latency, and cost;
- identify common operational and security failures.


# The analytics problem Athena solves

Data lakes may contain terabytes or petabytes of logs, events, extracts, and curated datasets in Amazon S3.

Traditional analytics often requires teams to:

- provision and size servers or clusters;
- load data into a separate database;
- patch and scale the query infrastructure;
- keep compute running even when no query is active.

Athena removes much of that infrastructure work and queries supported data sources using SQL.


# What is Amazon Athena?

Amazon Athena is a **serverless, interactive query service**.

- Submit SQL through the console, API, CLI, JDBC, or ODBC.
- Query data in place—commonly files in Amazon S3.
- Use table metadata from the AWS Glue Data Catalog or another supported catalog.
- Run distributed SQL without creating or managing a cluster.
- Pay primarily for query processing, commonly based on data scanned for SQL queries.

> Serverless means the service manages query infrastructure. It does not mean “no configuration,” “no limits,” or “free.”


# Where Athena fits

| Need | Athena fit |
|---|---|
| Interactive exploration of S3 data | Strong |
| SQL over service and application logs | Strong |
| Data-lake validation and ad hoc analysis | Strong |
| BI queries with suitable layout and governance | Strong |
| Transforming data with CTAS or INSERT INTO | Useful |
| Millisecond transactional reads/writes | Poor |
| High-frequency OLTP updates | Poor |
| Long, highly customized cluster workloads | Evaluate EMR or another engine |
| Repeated warehouse-style workloads with predictable performance | Compare with Redshift |


# The central mental model

```text
SQL client
   │
   ▼
Athena control plane ──► Workgroup policy and limits
   │
   ├──► Glue Data Catalog: schema, partitions, locations
   │
   ▼
Query planner ──► Distributed execution plan
   │
   ▼
Athena workers ◄──► Source data in S3
   │
   ▼
Query result files in S3 + execution metadata
```

**Metadata tells Athena how to interpret data. The source data normally remains in S3.**


# Four planes to keep separate

| Plane | Main resources | Responsibility |
|---|---|---|
| Data | S3 objects, partitions, table formats | Stores business data |
| Metadata | Catalogs, databases, tables, columns, partitions | Describes data and locations |
| Compute | Athena engine and distributed workers | Plans and executes SQL |
| Control/governance | Workgroups, IAM, Lake Formation, KMS, quotas | Controls access, settings, and usage |

Confusing these planes causes many design and troubleshooting errors.


# S3 as the data-lake foundation

S3 provides durable object storage for raw, refined, and curated layers.

Typical layout:

```text
s3://gksdatalake/
  bronze/orders/ingest_date=2026-09-01/...
  silver/orders/order_date=2026-08-31/...
  gold/daily_sales/year=2026/month=08/...
```

Athena reads objects; it does not turn S3 into a row-store database. File format, compression, object size, folder boundaries, and partition design strongly affect performance.


# The Glue Data Catalog relationship

Athena commonly uses `AwsDataCatalog`, backed by the AWS Glue Data Catalog.

- **Catalog**: top-level metadata source.
- **Database**: logical namespace for tables.
- **Table**: schema plus storage information; usually not the data itself.
- **Partition**: metadata for a subset, often mapped to an S3 prefix.
- **SerDe / format properties**: instructions for interpreting records.

A crawler can infer and register metadata, or engineers can define tables explicitly with DDL or infrastructure code.


# Schema-on-read

Athena applies a table definition when a query reads data.

```sql
SELECT order_id, customer_id, total_amount
FROM AwsDataCatalog.sales_curated.orders
WHERE order_date = DATE '2026-08-31';
```

The query depends on agreement among:

- the Catalog schema;
- the actual file structure and types;
- the table location and partition metadata;
- the reader/SerDe and file format.

Incorrect metadata can produce nulls, shifted columns, parsing errors, or silently incorrect results.


# Serverless does not mean stateless

You do not manage Athena servers, but persistent state still exists:

- source and transformed data in S3;
- table and partition metadata in a catalog;
- saved queries and named queries;
- workgroup settings and usage controls;
- query execution history and statistics;
- query result files in S3;
- permissions, encryption configuration, and audit logs.

AWS supplies and scales execution capacity; you remain responsible for data design, access, correctness, and cost controls.


# Workgroups: the operational boundary

An Athena workgroup separates workloads and applies shared settings.

It can govern or organize:

- query result location and encryption;
- engine version;
- per-query data-scan limits;
- metrics and usage visibility;
- tags and cost allocation;
- member access and workload isolation;
- whether workgroup settings override client-side settings.

Use separate workgroups for environments, teams, or workloads when their governance and cost boundaries differ.


# Query lifecycle: the big picture

```text
1. Submit SQL
2. Authorize and apply workgroup configuration
3. Parse and validate SQL
4. Resolve catalogs, tables, columns, and partitions
5. Build and optimize a logical plan
6. Create a distributed physical plan
7. Schedule stages and tasks on managed workers
8. Read, filter, join, aggregate, and exchange data
9. Materialize final results
10. Write result files to S3 and return status/statistics
```

Failures can occur before scanning, during execution, or while writing results.


# Step 1 — submission and authorization

A query request includes SQL, a workgroup, execution context, and possibly client-side result settings.

Before useful work begins, Athena must evaluate access such as:

- permission to start and inspect query executions;
- workgroup membership/use;
- access to the selected catalog and metadata;
- data permissions through IAM and, where used, Lake Formation;
- S3 and KMS permissions for sources and result output.

The principal starting a query and the service interactions involved must satisfy the applicable policies.


# Step 2 — parsing, analysis, and metadata resolution

The query planner first turns SQL text into a structured representation.

It then:

- checks syntax and statement support;
- resolves catalog, database, table, and column names;
- validates types and function calls;
- retrieves table, partition, and location metadata;
- determines candidate files or partitions;
- applies applicable governance decisions.

An error such as `TABLE_NOT_FOUND` or an invalid cast can stop the query before distributed data processing begins.


# Query planner: logical optimization

The planner expresses **what operations are required**, independent of individual worker tasks.

Common logical operations:

- table scan;
- projection—selecting required columns;
- filter;
- join;
- aggregation;
- sort, window, limit, and output.

Optimizations may include predicate pushdown, column pruning, constant folding, join reordering, and aggregation improvements. Available metadata and statistics affect the planner's choices.


# Cost-based optimization

Athena can use table and column statistics stored in the Glue Data Catalog.

Useful statistics include:

- row counts and data size;
- number of distinct values;
- null counts;
- minimum and maximum values.

With suitable statistics, the optimizer can compare alternative plans and choose a likely faster plan—for example, a better join order.

> Statistics improve decisions; they do not repair poor partitioning, tiny files, skew, or incorrect schemas.


# From logical plan to distributed plan

The planner divides work into **fragments or stages** connected by exchanges.

```text
Stage 2: final aggregation and output
             ▲
        network exchange
             ▲
Stage 1: partial aggregation / join
          ▲              ▲
Stage 0A: scan orders   Stage 0B: scan customers
```

Each stage can run as multiple tasks on managed nodes. Exchanges redistribute or gather intermediate data between stages.


# Query runtime

At runtime, Athena schedules tasks and processes data in parallel.

Workers may:

- enumerate and open source objects;
- decode and decompress file blocks;
- read only required column chunks when the format supports it;
- apply filters and projections;
- build and probe hash tables for joins;
- partially aggregate near the scan;
- exchange intermediate pages between stages;
- spill some intermediate data when memory pressure requires it;
- produce the final result stream.

Runtime speed depends on scanned bytes, parallelism, skew, file layout, and operator complexity—not only SQL length.


# Scan behavior and pushdown

Athena can avoid work at several levels:

| Technique | Avoids |
|---|---|
| Partition pruning | Entire partition prefixes |
| Column pruning | Unneeded columns or column chunks |
| Predicate pushdown | Some rows/row groups that cannot match |
| Compression | Physical bytes read and transferred |
| Columnar formats | Parsing unrelated fields |

```sql
SELECT customer_id, SUM(total_amount)
FROM sales_curated.orders
WHERE order_date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'
GROUP BY customer_id;
```

This is efficient only if layout, metadata, and types allow the engine to prune effectively.


# Joins and data exchange

A distributed hash join commonly has:

- a **build side**, ideally the smaller relation, used to build a hash table;
- a **probe side**, usually the larger relation, checked against that table.

Rows may be repartitioned across workers by the join key. Large build sides, skewed keys, and non-equality conditions increase memory, network, spill, and runtime pressure.

Athena can optimize joins, but it has less information about external files than a database that owns and continuously analyzes its storage.


# Inspecting the plan

Use `EXPLAIN` to view or validate a plan without executing the data scan:

```sql
EXPLAIN (TYPE DISTRIBUTED)
SELECT c.customer_name, SUM(o.total_amount)
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_name;
```

Use `EXPLAIN ANALYZE` to execute the query and include runtime statistics:

```sql
EXPLAIN ANALYZE
SELECT order_status, COUNT(*) FROM orders GROUP BY order_status;
```

`EXPLAIN ANALYZE` scans data and is billed like an executed query.


# Query states and timing

A query generally moves through states such as:

```text
QUEUED → RUNNING → SUCCEEDED
                 ↘ FAILED
                 ↘ CANCELLED
```

Execution statistics can separate:

- queue time;
- planning time;
- engine execution time;
- service processing time;
- total elapsed time;
- data scanned;
- result reuse status.

A slow query is not automatically a slow scan: it may be waiting, planning against many partitions, shuffling skewed data, or writing a large result.


# Where query results are stored

Athena SQL query results are written to an **S3 query-result location**.

The effective location can come from:

1. a workgroup configuration that enforces its settings;
2. a client/API request;
3. console configuration, where applicable.

The caller needs suitable access to the result bucket and, if encrypted with a customer-managed key, the applicable KMS permissions.

> A successful source read can still end in failure if Athena cannot write the result output.


# Three S3 locations—not one

| Location | Purpose | Example |
|---|---|---|
| Source table location | Files read by the query | `s3://gksdatalake/silver/orders/` |
| Query-result location | Result CSV/metadata written for executions | `s3://gks-athena-results/prod/` |
| CTAS/UNLOAD destination | New dataset deliberately produced by SQL | `s3://gksdatalake/gold/daily_sales/` |

Do not place routine query-result files inside a crawler's source prefix. Otherwise temporary output may be mistaken for business data.


# Result-location design

Production result storage should address:

- a dedicated bucket or controlled prefix per environment/workgroup;
- Block Public Access;
- least-privilege bucket and IAM policies;
- encryption, including KMS where required;
- lifecycle expiration for disposable results;
- logging and audit requirements;
- cross-account expected-bucket-owner controls where relevant;
- prevention of accidental crawler ingestion;
- Region and data-transfer implications.

Query history is not a substitute for a deliberate S3 retention policy.


# Result reuse: Athena's query-result cache

Athena can reuse a previously computed result when result reuse is enabled and an eligible matching query completed within the configured maximum age.

```text
New request
   │
   ├─ eligible matching result found ─► return existing result
   │
   └─ no eligible result ─► plan, scan, execute, write new result
```

Reuse can reduce latency and bytes scanned for repeated dashboard or exploratory queries.


# Cache semantics and boundaries

Result reuse is safest to understand as reuse of an earlier **query result**, not a general cache of S3 file blocks.

- It is opt-in/configured per query or client behavior.
- A maximum result age defines acceptable staleness.
- Eligibility depends on Athena's matching and compatibility checks.
- Changes in query text, context, catalog data, or unsupported query characteristics can prevent reuse.
- The previous result must remain accessible.
- Reuse favors speed and cost; freshness requirements may favor re-execution.

Always confirm the execution statistics rather than assuming a cache hit.


# Freshness versus reuse

Consider a dashboard refreshed every five minutes:

| Requirement | Sensible choice |
|---|---|
| Source changes hourly; 15-minute staleness accepted | Enable reuse with a bounded age |
| Financial control must include latest landed files | Disable reuse or use a very small age |
| Expensive immutable historical query repeated often | Reuse is attractive |
| Query depends on volatile or nondeterministic behavior | Do not rely on reused output |

Cache policy is a business-data freshness decision, not only a technical optimization.


# DDL, DML, and data-producing statements

| Category | Examples | Main effect |
|---|---|---|
| DDL | `CREATE DATABASE`, `CREATE EXTERNAL TABLE`, `ALTER TABLE` | Changes metadata |
| Read query | `SELECT` | Reads data and writes query output |
| CTAS | `CREATE TABLE AS SELECT` | Creates metadata and a new dataset |
| Insert | `INSERT INTO` | Adds output files to a target table |
| Export | `UNLOAD` | Writes query output in selected formats |
| Iceberg DML | `UPDATE`, `DELETE`, `MERGE` where supported | Changes transactional table state |

Engine version and table format determine exact statement support.


# CTAS: query plus transformation

```sql
CREATE TABLE sales_curated.orders_parquet
WITH (
  format = 'PARQUET',
  external_location = 's3://gksdatalake/silver/orders_parquet/',
  partitioned_by = ARRAY['order_year', 'order_month']
) AS
SELECT order_id,
       customer_id,
       CAST(total_amount AS DECIMAL(18,2)) AS total_amount,
       year(order_date) AS order_year,
       month(order_date) AS order_month
FROM sales_bronze.orders_csv;
```

CTAS can convert verbose row data into compressed columnar data. The dataset destination is separate from the ordinary query-result location.


# Performance begins with data design

High-impact practices:

- prefer Parquet or ORC for analytical tables;
- compress data with a splittable, supported codec where appropriate;
- partition by frequently filtered, reasonably sized dimensions;
- avoid thousands of tiny files;
- select required columns instead of `SELECT *`;
- filter early using correctly typed partition columns;
- materialize repeated expensive transformations;
- collect statistics where the optimizer can use them;
- use partition projection for suitable predictable partition schemes.


# Partition design: balance

Good partitions eliminate meaningful data without creating excessive metadata.

```text
Good candidate: order_date=2026-09-01/
Often too granular: customer_id=123456789/
```

Over-partitioning can cause:

- many tiny directories and files;
- expensive partition registration and planning;
- poor scan parallelism;
- operational complexity.

Under-partitioning makes queries scan irrelevant data. Choose partitions from real filter patterns and data volume.


# Cost model

For standard Athena SQL, cost is commonly driven by **bytes scanned**, subject to current regional pricing and minimums.

Related costs can include:

- S3 requests and storage for source and result objects;
- Glue Data Catalog storage/API usage;
- KMS requests;
- data transfer and federated source costs;
- logs, monitoring, and downstream BI tools;
- provisioned capacity if that consumption model is selected.

Compression, columnar formats, pruning, and reuse can lower both latency and scan cost.


# Security layers

Ask four separate questions:

1. **Can the identity use Athena and the workgroup?**
2. **Can it read metadata?** Glue Catalog and catalog permissions.
3. **Can it access data?** IAM/S3, KMS, and Lake Formation where enabled.
4. **Can it write/read results?** Result bucket and encryption-key permissions.

Also consider:

- bucket policies and access points;
- row, column, and cell filters through Lake Formation;
- CloudTrail and query auditability;
- secrets and network controls for federated connectors.


# Common failures and first checks

| Symptom | First checks |
|---|---|
| Table not found | Account, Region, catalog, database, spelling |
| Access denied reading data | IAM/Lake Formation, bucket policy, KMS key |
| Unable to verify/create output bucket | Result location, Region, S3 permissions |
| HIVE_BAD_DATA or parsing errors | Schema versus files, SerDe, mixed formats |
| Zero rows | Table location, partition registration, filter types |
| Too slow / too much scanned | Format, pruning, file sizes, selected columns |
| Resource exhausted | Join build side, skew, large sorts/windows, spill |
| Stale-looking answer | Result reuse age and source freshness |


# Observability and governance

Monitor more than success or failure:

- queue, planning, and engine execution times;
- bytes scanned and records processed;
- failed/cancelled query reasons;
- result reuse status;
- workgroup usage and scan-limit violations;
- expensive users, applications, and saved queries;
- source freshness and partition completeness;
- result-bucket growth and lifecycle expiration.

Tag resources, separate workgroups, publish cost expectations, and review high-scan queries regularly.


# Region and identity boundaries

Athena resources and the Glue Data Catalog are Region-aware.

- The same database or workgroup name can exist in multiple Regions.
- Console Region determines which resources and query history you see.
- S3 source and result buckets may introduce policy, latency, or transfer considerations across Regions.
- Operational evidence should record account, Region, workgroup, catalog, database, and query execution ID.

> A resource name alone is not a complete identity.


# End-to-end example

An analyst submits a monthly sales query:

1. The `finance_prod` workgroup enforces encrypted result output.
2. Athena authorizes the request and reads `sales_curated.orders` metadata.
3. The planner prunes all but the August partition and required Parquet columns.
4. Workers scan blocks in parallel and partially aggregate sales.
5. Exchange stages combine partial aggregates.
6. Athena writes the final result to the workgroup's S3 result prefix.
7. Statistics report elapsed time and bytes scanned.
8. A repeated eligible query may reuse that result within the permitted age.


# Architecture checklist

- Which account, Region, catalog, database, and workgroup?
- Where are source data, ordinary results, and transformed outputs stored?
- Who owns schema and partition correctness?
- Are file format, compression, file sizes, and partitions query-friendly?
- Which principal can read metadata, source data, and result data?
- Is KMS or Lake Formation involved?
- Does the workgroup enforce location, encryption, limits, and engine version?
- Is result reuse compatible with freshness requirements?
- What are the expected scanned bytes, concurrency, and cost?
- What evidence proves correctness, freshness, and completion?


# Common misconceptions

| Misconception | Correction |
|---|---|
| Athena stores the data | Source data usually remains in S3 or another connected source |
| Glue tables contain rows | They primarily contain metadata describing datasets |
| Serverless means unlimited | Quotas, capacity, memory, and concurrency still matter |
| A successful query proves the schema is correct | Incorrect schema can produce plausible but wrong results |
| `LIMIT 10` always scans only ten rows | Scan reduction depends on plan, format, and layout |
| Query results and CTAS data use the same destination | They are distinct output concepts |
| Cache means S3 blocks are permanently cached | Result reuse returns an eligible prior query result |


# Knowledge check

1. Which component stores a table's S3 location and column definitions?
2. Why can planning time be high before much data is scanned?
3. What is the difference between a logical plan and a distributed plan?
4. Why does a hash join care about the build-side size?
5. Where does Athena write ordinary query results?
6. How is that location different from a CTAS destination?
7. When can result reuse be harmful to business correctness?
8. Name three ways to reduce bytes scanned.
9. Which permissions are needed when both source and result objects use KMS?
10. Why should a crawler avoid the Athena results prefix?


# Recap

**Athena connects six ideas:**

1. S3 or another supported source stores the data.
2. A catalog describes how to find and interpret it.
3. A workgroup governs query behavior and usage.
4. The planner converts SQL into an optimized distributed plan.
5. The runtime scans and processes data on managed workers.
6. Results are persisted to S3 and may be reused when eligible.

> The shortest accurate model: **Athena is a serverless distributed SQL service that separates storage, metadata, compute, and query-result persistence.**


# Official references

- [What is Amazon Athena?](https://docs.aws.amazon.com/athena/latest/ug/what-is.html)
- [Use Athena SQL](https://docs.aws.amazon.com/athena/latest/ug/using-athena-sql.html)
- [Tables, databases, and catalogs](https://docs.aws.amazon.com/athena/latest/ug/understanding-tables-databases-and-the-data-catalog.html)
- [Specify a query result location](https://docs.aws.amazon.com/athena/latest/ug/querying.html)
- [Reuse query results](https://docs.aws.amazon.com/athena/latest/ug/reusing-query-results.html)
- [EXPLAIN and EXPLAIN ANALYZE](https://docs.aws.amazon.com/athena/latest/ug/athena-explain-statement.html)
- [Understand EXPLAIN results](https://docs.aws.amazon.com/athena/latest/ug/athena-explain-statement-understanding.html)
- [Cost-based optimizer](https://docs.aws.amazon.com/athena/latest/ug/cost-based-optimizer.html)
- [Optimize queries](https://docs.aws.amazon.com/athena/latest/ug/performance-tuning-query-optimization-techniques.html)
- [Athena security](https://docs.aws.amazon.com/athena/latest/ug/security.html)
